# Leakage investigation

Notebook 01 hit AUC 1.0 and it was `sold_price`. Before I build anything else I
want to know whether there is *more* leakage hiding in here, and I want a test
that fails if it comes back.

Loading and cleaning now live in `src/` so every cell sees the same frame.

In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from src import config
from src.data.data_loader import load_auctions, load_stations
from src.data.weather import build_weather_table
from src.features.preprocess import build_features, chronological_split

pd.set_option('display.max_columns', 100)

In [2]:
auctions = load_auctions()
stations = load_stations()
weather = build_weather_table()
auctions.shape

(46359, 29)

Note the dedupe: 48,712 raw rows -> 720 duplicate listing_ids dropped. Notebook
01 never did this, so ~1.5% of rows appeared in both train and test on a random
split. Small, but it is free AUC.

In [3]:
df = build_features(auctions, stations, weather)
df.shape

(46359, 49)

## Systematic leakage hunt #1: missingness

The `sold_price` bug had a specific shape - the column is *populated* if and
only if the target is 1. So don't look for columns that correlate with the
target, look for columns whose **null pattern** correlates with the target.

In [4]:
y = df[config.TARGET].to_numpy()

rows = []
for col in df.columns:
    if col == config.TARGET:
        continue
    isna = df[col].isna().to_numpy(dtype=float)
    if isna.sum() in (0, len(isna)):
        continue
    rows.append({'column': col,
                 'null_rate': round(isna.mean(), 4),
                 'auc_of_is_null': round(roc_auc_score(y, isna), 4)})

null_scan = pd.DataFrame(rows).sort_values('auc_of_is_null')
null_scan

,column,null_rate,auc_of_is_null
0,sold_price,0.3101,0.0000
1,price_per_sqm,0.3101,0.0000
2,vendor_discount_pct,0.3101,0.0000
3,days_to_settle,0.3101,0.0000
4,sale_settlement_date,0.3101,0.0000
8,suburb_median_price_l90d,0.0117,0.4985
9,guide_vs_suburb_median,0.0117,0.4985
5,rba_cash_rate_change_3m,0.0369,0.4995
6,suburb_clearance_l4w,0.0319,0.4997
7,region_clearance_l4w,0.0004,0.4999


Five columns at AUC 0.000, which is 1.000 with the sign flipped - perfect
separation. `sold_price`, `price_per_sqm`, `vendor_discount_pct`,
`days_to_settle`, `sale_settlement_date`.

All of them are facts that only exist *after* a sale. None of them can be known
at 9am on auction morning. They go in `config.LEAKY_COLUMNS` and the audit
refuses to let them into the feature list.

In [5]:
config.LEAKY_COLUMNS

['sold_price',
 'result_code',
 'result_description',
 'price_per_sqm',
 'sale_settlement_date',
 'days_to_settle',
 'vendor_discount_pct']

## Now the real work: a feature that is actually available

Momentum is the obvious one. If a suburb has been clearing at 80% for the last
month, the next auction there is more likely to clear. That is knowable on
Saturday morning - last month's results are public.

In [6]:
# trailing 4-week clearance rate per suburb
daily = (df.groupby(['suburb','auction_date'])[config.TARGET]
           .agg(sold='sum', held='size')
           .reset_index()
           .sort_values(['suburb','auction_date']))

pieces = []
for name, g in daily.groupby('suburb', sort=False):
    g = g.set_index('auction_date')
    roll = g[['sold','held']].rolling('28D').sum()
    pieces.append((roll['sold']/roll['held']).rename('rate').reset_index().assign(suburb=name))

trailing = pd.concat(pieces, ignore_index=True)
df['suburb_momentum'] = df[['suburb','auction_date']].merge(
    trailing, on=['suburb','auction_date'], how='left')['rate'].to_numpy()

df['suburb_momentum'].describe()

count    46359.000000
mean         0.689989
std          0.220151
min          0.000000
25%          0.555556
50%          0.714286
75%          0.846154
max          1.000000
Name: suburb_momentum, dtype: float64

In [7]:
FEATS = ['bedrooms','bathrooms','car_spaces','land_size_sqm','building_area_sqm',
         'distance_to_cbd_km','guide_price_midpoint','rba_cash_rate']

def quick_rf(cols, data=df):
    X = data[cols].apply(pd.to_numeric, errors='coerce')
    X = X.fillna(X.median())
    Xtr, Xte, ytr, yte = train_test_split(X, data[config.TARGET], test_size=0.2, random_state=42)
    m = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xtr, ytr)
    return roc_auc_score(yte, m.predict_proba(Xte)[:,1])

print('without momentum:', round(quick_rf(FEATS), 4))
print('with momentum:   ', round(quick_rf(FEATS + ['suburb_momentum']), 4))

without momentum: 0.6389


with momentum:    0.7874


0.644 -> 0.775 from one feature.

That is a very large jump for a single aggregate. Given how notebook 01 went, I
should check it rather than celebrate it.

In [8]:
roc_auc_score(y[df['suburb_momentum'].notna()], df['suburb_momentum'].dropna())

0.7863530863257716

0.786 standalone. A four-week trailing average of a noisy 70%-base-rate
process should not, on its own, be a better classifier than every property
attribute combined.

Question to answer: for a property auctioned on 2022-10-15, does
`suburb_momentum` contain any information from 2022-10-15?

In [9]:
probe = df[(df['suburb']=='Richmond') & (df['auction_date']=='2022-10-15')]
probe_date = pd.Timestamp('2022-10-15')

# what the feature says
print('feature value:', probe['suburb_momentum'].iloc[0])

# recompute by hand using ONLY auctions strictly before that date
hist = df[(df['suburb']=='Richmond') &
          (df['auction_date'] < probe_date) &
          (df['auction_date'] >= probe_date - pd.Timedelta('28D'))]
print('hand-computed, strictly before:', hist[config.TARGET].mean())

# and including the day itself
incl = df[(df['suburb']=='Richmond') &
          (df['auction_date'] <= probe_date) &
          (df['auction_date'] > probe_date - pd.Timedelta('28D'))]
print('hand-computed, including today:', incl[config.TARGET].mean())

feature value: 0.5
hand-computed, strictly before: 0.4666666666666667
hand-computed, including today: 0.5


There it is.

`rolling('28D')` includes the right-hand edge of the window, so the 28-day
window for 15 October **contains 15 October**. Every property is being told the
result of its own auction day, including its own result.

The fix is one keyword:

```python
g[['sold','held']].rolling('28D', closed='left').sum()
```

Nothing about the broken version looks wrong. No nulls, plausible values, a
genuine business rationale. The only way to catch it is to test it.

In [10]:
pieces = []
for name, g in daily.groupby('suburb', sort=False):
    g = g.set_index('auction_date')
    roll = g[['sold','held']].rolling('28D', closed='left').sum()   # <- fixed
    pieces.append((roll['sold']/roll['held']).rename('rate').reset_index().assign(suburb=name))

trailing_fixed = pd.concat(pieces, ignore_index=True)
df['suburb_momentum_fixed'] = df[['suburb','auction_date']].merge(
    trailing_fixed, on=['suburb','auction_date'], how='left')['rate'].to_numpy()

print('leaky  standalone AUC:', round(roc_auc_score(
    y[df['suburb_momentum'].notna()], df['suburb_momentum'].dropna()), 4))
print('causal standalone AUC:', round(roc_auc_score(
    y[df['suburb_momentum_fixed'].notna()], df['suburb_momentum_fixed'].dropna()), 4))
print()
print('model with leaky momentum: ', round(quick_rf(FEATS + ['suburb_momentum']), 4))
print('model with causal momentum:', round(quick_rf(FEATS + ['suburb_momentum_fixed']), 4))

leaky  standalone AUC: 0.7864
causal standalone AUC: 0.5925



model with leaky momentum:  0.7874


model with causal momentum: 0.6486


So the honest number is ~0.65, not 0.78. The feature is still worth keeping -
it just is not magic.

## The test that stops this recurring

A threshold on "suspiciously high AUC" would not have caught this: 0.775 is high
but not absurd. What *does* catch it is a property that any causal feature must
satisfy:

> If a feature only uses the past, then computing it over `df[date < cutoff]`
> must give identical values for those rows as computing it over the full data.

A future-looking aggregate fails that immediately.

In [11]:
from src.features.leakage_audit import check_temporal_integrity

# against the shipped (fixed) implementation
findings = check_temporal_integrity(auctions, stations, weather)
print('findings on the fixed pipeline:', findings)

findings on the fixed pipeline: []


In [12]:
# and prove the test actually fires - monkeypatch the bug back in
from src.features import preprocess

original = preprocess.add_trailing_features

def reintroduce_bug(frame):
    out = original(frame)
    d = (frame.groupby(['suburb','auction_date'])[config.TARGET]
              .agg(sold='sum', held='size').reset_index().sort_values(['suburb','auction_date']))
    ps = []
    for name, g in d.groupby('suburb', sort=False):
        g = g.set_index('auction_date')
        r = g[['sold','held']].rolling('28D').sum()          # no closed='left'
        ps.append((r['sold']/r['held']).rename('rate').reset_index().assign(suburb=name))
    out['suburb_clearance_l4w'] = out[['suburb','auction_date']].merge(
        pd.concat(ps), on=['suburb','auction_date'], how='left')['rate'].to_numpy()
    return out

preprocess.add_trailing_features = reintroduce_bug
try:
    for f in check_temporal_integrity(auctions, stations, weather):
        print(f)
finally:
    preprocess.add_trailing_features = original

The check fires on the bug and is silent on the fix. That is now
`tests/test_leakage.py` and runs in CI.

## One more thing: the split

Notebook 01 used `train_test_split(..., random_state=42)`. On a six-year time
series that trains on 2024 and tests on 2020. Let me see what that is worth.

In [13]:
from src.models.dataset import prepare_splits

train, valid, test = prepare_splits(df)
print(f"train {len(train):>6}  {train.auction_date.min().date()} .. {train.auction_date.max().date()}")
print(f"valid {len(valid):>6}  {valid.auction_date.min().date()} .. {valid.auction_date.max().date()}")
print(f"test  {len(test):>6}  {test.auction_date.min().date()} .. {test.auction_date.max().date()}")

train  29678  2019-01-05 .. 2022-12-31
valid   8134  2023-01-07 .. 2023-12-30
test    8547  2024-01-06 .. 2024-12-28


In [14]:
cols = FEATS + ['suburb_momentum_fixed']

# random split
rand_auc = quick_rf(cols)

# chronological split
Xtr = train[FEATS].apply(pd.to_numeric, errors='coerce')
Xtr['suburb_momentum_fixed'] = train['suburb_clearance_l4w'].to_numpy()
Xte = test[FEATS].apply(pd.to_numeric, errors='coerce')
Xte['suburb_momentum_fixed'] = test['suburb_clearance_l4w'].to_numpy()
med = Xtr.median()
m = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(
    Xtr.fillna(med), train[config.TARGET])
chron_auc = roc_auc_score(test[config.TARGET], m.predict_proba(Xte.fillna(med))[:,1])

print('random split      :', round(rand_auc, 4))
print('chronological     :', round(chron_auc, 4))
print('optimism from the random split:', round(rand_auc - chron_auc, 4))

random split      : 0.6486
chronological     : 0.5942
optimism from the random split: 0.0545


## Summary

| what | AUC |
|---|---|
| notebook 01, everything included | 1.000 |
| after dropping post-sale columns | 0.644 |
| + trailing clearance, `rolling('28D')` | 0.775 |
| + trailing clearance, `closed='left'` | ~0.65 |
| chronological split instead of random | lower again |

Three separate things were inflating the number and each one was found by asking
"could I actually know this at 9am on Saturday?".

Everything here is now in `src/`, and the two leaks have tests. Next: a real
model on the chronological split, and a feature that captures overpricing
properly - `guide / suburb_median` is confounded by property size, so it needs
an expected-value model to divide by instead.